# 🛡️ Multimodal Face Liveness Model Training Workflow
This notebook handles the entire end-to-end training process for the binary face anti-spoofing classifier on your GPU/ROCm-accelerated training server. 

### Automated Workflow Steps:
1. **Verify ROCm Hardware**: Checks and initializes PyTorch ROCm/GPU execution.
2. **Dataset Acquisition & Fallback**: Downloads LFW dataset from Kaggle, falling back to HTTP mirror or synthetic generation.
3. **Advanced Spoof Engineering**: Programmatically synthesizes highly realistic spoofing attacks (LCD Moire grids, print reflection, pixelation, and mock screen bezels) from a subset of face images.
4. **Data Splitting & Loading**: Structures train/validation datasets and applies robust data augmentations.
5. **Model Compilation**: Imports pre-trained MobileNetV3 and adapts the classifier head.
6. **ROCm Training & Validation**: Trains the classifier, logs metric curves, and saves production model weights (`liveness_model.pt`).

In [ ]:
!pip install matplotlib kaggle

In [ ]:
import os
import tarfile
import urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt

print("PyTorch Version:", torch.__version__)
print("ROCm/CUDA GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

## 1. Automated Dataset Acquisition
We download the Labeled Faces in the Wild (LFW) dataset using the Kaggle API. If Kaggle download is blocked or unavailable, we automatically fall back to direct HTTP mirror downloading.

In [ ]:
# Create workspace directories
os.makedirs("liveness_dataset", exist_ok=True)
dataset_dir = "liveness_dataset"
extract_path = os.path.join(dataset_dir, "extracted")

# Pre-configure Kaggle Token environment variables
os.environ["KAGGLE_API_TOKEN"] = "KGAT_ebb30aa62041e8cbdee1192e8901ce62"
try:
    home_kaggle = os.path.expanduser("~/.kaggle")
    os.makedirs(home_kaggle, exist_ok=True)
    with open(os.path.join(home_kaggle, "access_token"), "w") as f:
        f.write("KGAT_ebb30aa62041e8cbdee1192e8901ce62\n")
    os.chmod(os.path.join(home_kaggle, "access_token"), 0o600)
except Exception as e:
    print(f"Kaggle credentials configuration warning: {e}")

download_completed = False

# Primary option: Kaggle Download
if not os.path.exists(extract_path) or len(os.listdir(extract_path)) == 0:
    print("Attempting dataset download via Kaggle API...")
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files('jessicali9530/lfw-dataset', path=extract_path, unzip=True)
        print("Download and extraction completed via Kaggle!")
        download_completed = True
    except Exception as e:
        print(f"Kaggle API download failed: {e}. Trying direct HTTP mirror fallback...")
        
        # Secondary option: Direct HTTP Mirror
        lfw_url = "http://vis-www.cs.umass.edu/lfw/lfw-funneled.tgz"
        tar_path = os.path.join(dataset_dir, "lfw-funneled.tgz")
        try:
            if not os.path.exists(tar_path):
                urllib.request.urlretrieve(lfw_url, tar_path)
            print("HTTP download completed. Extracting...")
            with tarfile.open(tar_path, "r") as tar:
                tar.extractall(path=extract_path)
            print("Extraction completed via HTTP!")
            download_completed = True
        except Exception as http_err:
            print(f"HTTP mirror fallback failed: {http_err}.")

## 2. Advanced Feature Engineering & Spoof Data Synthesis
To train a classifier, we need both positive (real) and negative (spoof) examples. Since academic databases are often restricted, we engineer fakes from real images using physics-informed transformations:
- **Moire Grid Effect**: Simulates capturing an LCD monitor or phone screen.
- **Print Specular Glare**: Blends a light source overlay to model reflective paper print attacks.
- **Pixelation & Resolution Degradation**: Models typical reprint resolution loss.
- **Screen Borders**: Adds a phone/tablet bezel border around the crop.

In [ ]:
def synthesize_spoof(img_path: str) -> Image.Image:
    """Applies physics-informed print and screen-replay transformations to simulate spoof attacks."""
    img = Image.open(img_path).convert("RGB")
    width, height = img.size
    
    # 1. Specular Glare / Reflection (Print Attack)
    if np.random.rand() > 0.5:
        glare = Image.new("RGB", (width, height), color=(255, 255, 255))
        draw = ImageDraw.Draw(glare)
        cx, cy = np.random.randint(50, width-50), np.random.randint(50, height-50)
        radius = np.random.randint(30, 80)
        for r in range(radius, 0, -2):
            color = int(255 * (1.0 - r/radius))
            draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=(color, color, color))
        img = Image.blend(img, glare, alpha=0.3)
        
    # 2. Moire Pattern / Scanlines (Screen Replay Attack)
    if np.random.rand() > 0.5:
        grid = np.zeros((height, width, 3), dtype=np.uint8)
        for y in range(height):
            for x in range(width):
                val = int(127 + 127 * np.sin(x * 1.5) * np.sin(y * 1.5))
                grid[y, x] = [val, val, val]
        grid_img = Image.fromarray(grid)
        img = Image.blend(img, grid_img, alpha=0.15)
        
    # 3. Blur & Contrast degradation (Reprint/Replay Loss of detail)
    img = img.filter(ImageFilter.GaussianBlur(radius=np.random.uniform(0.5, 1.8)))
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(np.random.uniform(0.7, 1.4))
    
    # 4. Device Bezel border
    if np.random.rand() > 0.6:
        draw = ImageDraw.Draw(img)
        draw.rectangle([0, 0, width-1, height-1], outline=(10, 10, 10), width=15)
        
    return img

### Visualizing the Data Engineering Pipeline
Let's draw a genuine face alongside its synthesized spoof counterpart to verify visual authenticity.

In [ ]:
# Find a sample image
all_images = []
for root, dirs, files in os.walk(extract_path):
    for f in files:
        if f.endswith(".jpg"):
            all_images.append(os.path.join(root, f))

if all_images:
    sample_img_path = all_images[0]
    spoof_img = synthesize_spoof(sample_img_path)
    real_img = Image.open(sample_img_path)
else:
    # Create a dummy image for visualization fallback
    print("No downloaded images found. Creating a synthetic visual sample...")
    real_img = Image.new("RGB", (250, 250), color=(180, 220, 240))
    draw = ImageDraw.Draw(real_img)
    draw.ellipse([50, 50, 200, 200], fill=(240, 200, 180))
    draw.ellipse([85, 95, 105, 115], fill=(0, 0, 0))
    draw.ellipse([145, 95, 165, 115], fill=(0, 0, 0))
    draw.arc([90, 130, 160, 170], start=0, end=180, fill=(0, 0, 0), width=3)
    
    # Save dummy image temporarily to feed to synthesize_spoof
    os.makedirs("liveness_dataset", exist_ok=True)
    temp_path = "liveness_dataset/temp_vis_sample.jpg"
    real_img.save(temp_path)
    spoof_img = synthesize_spoof(temp_path)
    if os.path.exists(temp_path):
        os.remove(temp_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(real_img)
axes[0].set_title("Genuine Face (Class: Real)")
axes[0].axis("off")

axes[1].imshow(spoof_img)
axes[1].set_title("Synthesized Spoof (Class: Spoof)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 3. Dataset Structuring (Train / Validation Splits)
We split LFW images into a balanced train/val directory structure. If LFW isn't available, we automatically fallback to mock_dataset and create synthetic images.

In [ ]:
import shutil
import random

train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")

# Reset directories
for path in [train_dir, val_dir]:
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(os.path.join(path, "real"), exist_ok=True)
    os.makedirs(os.path.join(path, "spoof"), exist_ok=True)

# Populate train & val splits
if len(all_images) >= 1250:
    train_samples = all_images[:1000]
    val_samples = all_images[1000:1250]

    print("Preparing train split...")
    for idx, src_path in enumerate(train_samples):
        if idx % 2 == 0:
            shutil.copy(src_path, os.path.join(train_dir, "real", f"real_{idx}.jpg"))
        else: 
            spoof = synthesize_spoof(src_path)
            spoof.save(os.path.join(train_dir, "spoof", f"spoof_{idx}.jpg"))

    print("Preparing validation split...")
    for idx, src_path in enumerate(val_samples):
        if idx % 2 == 0:
            shutil.copy(src_path, os.path.join(val_dir, "real", f"real_{idx}.jpg"))
        else:
            spoof = synthesize_spoof(src_path)
            spoof.save(os.path.join(val_dir, "spoof", f"spoof_{idx}.jpg"))
else:
    print("LFW not available. Using mock_dataset and generating synthetic samples...")
    def create_synthetic_image(save_path, is_spoof=False):
        img = Image.new("RGB", (224, 224), color=(random.randint(50, 200), random.randint(50, 200), random.randint(50, 200)))
        draw = ImageDraw.Draw(img)
        # Draw circle
        draw.ellipse([50, 50, 174, 174], fill=(random.randint(200, 255), random.randint(150, 200), random.randint(100, 150)))
        # Draw eyes
        draw.ellipse([80, 90, 100, 110], fill=(0, 0, 0))
        draw.ellipse([124, 90, 144, 110], fill=(0, 0, 0))
        # Mouth
        draw.arc([80, 120, 144, 150], start=0, end=180, fill=(0, 0, 0), width=3)
        if is_spoof:
            glare = Image.new("RGB", (224, 224), color=(255, 255, 255))
            glare_draw = ImageDraw.Draw(glare)
            glare_draw.ellipse([40, 40, 120, 120], fill=(200, 200, 200))
            img = Image.blend(img, glare, alpha=0.3)
            # border
            draw = ImageDraw.Draw(img)
            draw.rectangle([0, 0, 223, 223], outline=(10, 10, 10), width=15)
        img.save(save_path)

    # Generate 100 train real, 100 train spoof
    for i in range(100):
        create_synthetic_image(os.path.join(train_dir, "real", f"synthetic_real_{i}.jpg"), is_spoof=False)
        create_synthetic_image(os.path.join(train_dir, "spoof", f"synthetic_spoof_{i}.jpg"), is_spoof=True)
    
    # Generate 25 val real, 25 val spoof
    for i in range(25):
        create_synthetic_image(os.path.join(val_dir, "real", f"synthetic_real_{i}.jpg"), is_spoof=False)
        create_synthetic_image(os.path.join(val_dir, "spoof", f"synthetic_spoof_{i}.jpg"), is_spoof=True)

abs_dataset_path = os.path.abspath(dataset_dir)
print(f"Data splits ready. Train size: {len(os.listdir(os.path.join(train_dir, 'real'))) * 2}, Val size: {len(os.listdir(os.path.join(val_dir, 'real'))) * 2}")
print(f"Absolute dataset directory: {abs_dataset_path}")

## 4. Training Pipeline & Model Definition
We define our data transformations, build PyTorch data loaders, compile our adapted MobileNetV3 classifier head, and execute GPU training.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

class LivenessModel(nn.Module):
    def __init__(self):
        super(LivenessModel, self).__init__()
        self.backbone = models.mobilenet_v3_small(pretrained=True)
        in_features = self.backbone.classifier[3].in_features
        self.backbone.classifier[3] = nn.Linear(in_features, 2)

    def forward(self, x):
        return self.backbone(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Target training device:", device)

model = LivenessModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Run Training Loop (ROCm Accelerated)
Let's run 5 epochs of training to reach excellent performance on this dataset.

In [ ]:
epochs = 5
print("Starting Face Liveness model training...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct / total) * 100
    
    # Validation Loop
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_epoch_loss = val_loss / len(val_loader.dataset)
    val_acc = (val_correct / val_total) * 100
    
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f} Acc: {val_acc:.2f}%")

# Save model weights to the target locations
save_paths = ["liveness_model.pt"]
try:
    if os.path.basename(os.getcwd()) == "notebooks":
        save_paths.append("../notebooks/liveness_model.pt")
    else:
        save_paths.append("notebooks/liveness_model.pt")
except Exception:
    pass

for p in save_paths:
    try:
        torch.save(model.state_dict(), p)
        print(f"Model saved successfully to: {p}")
    except Exception as e:
        print(f"Could not save to {p}: {e}")

## 5. Visualizing Classifier Predictions
We run inference on validation images to visualize liveness detection classification output.

In [ ]:
# Load one batch from val loader
inputs, labels = next(iter(val_loader))
model.eval()
with torch.no_grad():
    outputs = model(inputs.to(device))
    _, preds = outputs.max(1)

# Helper to un-normalize and show image
def show_img(inp, title=None, color="black"):
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title:
        plt.title(title, color=color, fontsize=10)
    plt.axis("off")

# Plot predictions
fig = plt.figure(figsize=(12, 8))
classes = ["Real", "Spoof"]
num_to_plot = min(6, len(inputs))

for i in range(num_to_plot):
    ax = fig.add_subplot(2, 3, i + 1)
    true_label = classes[labels[i].item()]
    pred_label = classes[preds[i].item()]
    is_correct = (labels[i].item() == preds[i].item())
    
    title = f"True: {true_label}\nPred: {pred_label}"
    color = "green" if is_correct else "red"
    show_img(inputs[i], title=title, color=color)

plt.tight_layout()
plt.show()